# 02 — Financial Ratio Analysis & Three-Period Trend Review

## Purpose
Calculate 20+ credit-relevant ratios and assess 3-period trends with automated flags.

**Bank context:** After spreading, the analyst calculates ratios across all periods and looks for improving/deteriorating trends. This maps to the **Derived_Ratios** sheet of the AU SME Borrower Model.

**Ratio categories:**
- Profitability (margins, returns)
- Leverage (debt capacity)
- Coverage — the "Four Measures of Capacity" from Commercial Ready: ICR, DSCR, Payback Ratio, Multiple of Earnings
- Liquidity (current ratio, quick ratio)
- Efficiency (debtor days, inventory days)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_sme_dataset
from src.ratio_engine import calculate_ratios, borrower_ratio_summary, BANK_THRESHOLDS
from src.trend_analysis import analyse_trends, portfolio_trend_summary

pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

# Load and calculate ratios
df = generate_sme_dataset(n_borrowers=80, seed=42)
df_r = calculate_ratios(df)
print(f'Calculated {len([c for c in df_r.columns if c not in df.columns])} ratio columns')

## Reference Borrower — Ratio Summary

All ratios for Example AU SME Pty Ltd across 3 periods.

In [ ]:
summary = borrower_ratio_summary(df_r, borrower_id=0)

# Add bank thresholds for context
summary['threshold'] = ''
for metric, info in BANK_THRESHOLDS.items():
    if metric in summary.index:
        summary.loc[metric, 'threshold'] = info['pass']

print('='*70)
print('FINANCIAL RATIOS — Example AU SME Pty Ltd')
print('='*70)
display(summary)

## Three-Period Trend Analysis

Banks require trend analysis across at least 3 periods. Each metric is classified as:
- **POSITIVE** — improving trend (or declining for leverage ratios)
- **NEGATIVE** — deteriorating trend
- **FLAT** — broadly stable

This maps to the `Derived_Ratios` sheet rows 23-40.

In [ ]:
trends = analyse_trends(df_r, borrower_id=0)

# Colour-code the status
def colour_status(val):
    if val == 'POSITIVE':
        return 'background-color: #C8E6C9'  # green
    elif val == 'NEGATIVE':
        return 'background-color: #FFCDD2'  # red
    return 'background-color: #FFF9C4'  # yellow

print('='*70)
print('THREE-PERIOD TREND REVIEW — Example AU SME Pty Ltd')
print(f'Core flags passed: {trends["pass"].sum()} / {len(trends)}')
print('='*70)

display(
    trends[['metric', 'FY-2', 'FY-1', 'FY0', 'slope', 'status', 'comment']]
    .style.applymap(colour_status, subset=['status'])
)

## Key Ratio Trend Charts

Visual representation of the most critical ratios — the kind of charts included in credit committee papers.

In [ ]:
ref = df_r[df_r['borrower_id'] == 0].sort_values('period')
periods = ['FY-2', 'FY-1', 'FY0']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# ICR
axes[0, 0].plot(periods, ref['icr'], 'o-', linewidth=2, color='#2196F3', markersize=8)
axes[0, 0].axhline(2.0, color='red', linestyle='--', alpha=0.7, label='Threshold: 2.0x')
axes[0, 0].set_title('ICR (EBIT / Interest)', fontweight='bold')
axes[0, 0].set_ylabel('Times')
axes[0, 0].legend(fontsize=9)

# DSCR
axes[0, 1].plot(periods, ref['dscr'], 'o-', linewidth=2, color='#4CAF50', markersize=8)
axes[0, 1].axhline(1.20, color='red', linestyle='--', alpha=0.7, label='Threshold: 1.20x')
axes[0, 1].set_title('DSCR (OCF / Debt Service)', fontweight='bold')
axes[0, 1].set_ylabel('Times')
axes[0, 1].legend(fontsize=9)

# Debt / EBITDA
axes[0, 2].plot(periods, ref['debt_to_ebitda'], 'o-', linewidth=2, color='#FF9800', markersize=8)
axes[0, 2].axhline(4.0, color='red', linestyle='--', alpha=0.7, label='Threshold: 4.0x')
axes[0, 2].set_title('Debt / EBITDA', fontweight='bold')
axes[0, 2].set_ylabel('Times')
axes[0, 2].legend(fontsize=9)

# Current Ratio
axes[1, 0].plot(periods, ref['current_ratio'], 'o-', linewidth=2, color='#9C27B0', markersize=8)
axes[1, 0].axhline(1.20, color='red', linestyle='--', alpha=0.7, label='Threshold: 1.20x')
axes[1, 0].set_title('Current Ratio', fontweight='bold')
axes[1, 0].set_ylabel('Times')
axes[1, 0].legend(fontsize=9)

# Free Cash Flow
axes[1, 1].bar(periods, ref['free_cash_flow'] / 1e6, color='#00BCD4', alpha=0.8)
axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.7)
axes[1, 1].set_title('Free Cash Flow (A$M)', fontweight='bold')
axes[1, 1].set_ylabel('A$ Millions')

# FCCR
axes[1, 2].plot(periods, ref['fccr'], 'o-', linewidth=2, color='#E91E63', markersize=8)
axes[1, 2].axhline(1.20, color='red', linestyle='--', alpha=0.7, label='Threshold: 1.20x')
axes[1, 2].set_title('FCCR', fontweight='bold')
axes[1, 2].set_ylabel('Times')
axes[1, 2].legend(fontsize=9)

fig.suptitle('Key Credit Ratios — Example AU SME Pty Ltd', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/02_ratio_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## Portfolio-Level Trend Summary

How many borrowers show predominantly positive trends?

In [ ]:
port_trends = portfolio_trend_summary(df_r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of positive flag %
axes[0].hist(port_trends['positive_pct'] * 100, bins=15, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribution of Positive Trend Flags (%)', fontweight='bold')
axes[0].set_xlabel('% of Metrics Showing Positive Trend')
axes[0].set_ylabel('Number of Borrowers')
axes[0].axvline(port_trends['positive_pct'].median() * 100, color='red', linestyle='--',
                label=f'Median: {port_trends["positive_pct"].median()*100:.0f}%')
axes[0].legend()

# Negative flags vs positive flags
axes[1].scatter(port_trends['positive_flags'], port_trends['negative_flags'],
                alpha=0.6, color='#2196F3', s=40)
axes[1].set_title('Positive vs Negative Flags by Borrower', fontweight='bold')
axes[1].set_xlabel('Positive Flags')
axes[1].set_ylabel('Negative Flags')

plt.tight_layout()
plt.savefig('../outputs/figures/02_portfolio_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Portfolio Trend Summary:')
print(f'  Median positive flags: {port_trends["positive_flags"].median():.0f} / {port_trends["total_metrics"].iloc[0]}')
print(f'  Borrowers with >70% positive: {(port_trends["positive_pct"] > 0.70).sum()}')
print(f'  Borrowers with <30% positive: {(port_trends["positive_pct"] < 0.30).sum()}')

---

## Key Takeaways

1. **Coverage ratios are the priority** — ICR >= 2.0x, DSCR >= 1.20x are typical bank minimums
2. **Leverage thresholds** — Debt/EBITDA <= 4.0x is a standard covenant level
3. **Trend direction matters as much as absolute level** — an improving borrower at 1.5x DSCR may be preferable to a declining borrower at 2.0x
4. **Automated trend flags** reduce subjectivity and ensure consistent assessment

**Next:** [03_Working_Capital_Deep_Dive.ipynb](03_Working_Capital_Deep_Dive.ipynb)